# Website visit summary
Full local history · rerun after exporting new events

In [ ]:
import io
import json
import os
import re
import html
import ipaddress
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import unquote

import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from IPython.display import HTML, Image, Markdown, display

TIMEZONE = ZoneInfo('America/Chicago')
VISIT_GAP_MINUTES = 30
OWNER_EMAIL = 'rekarpyrce@gmail.com'
# Override only the file location, never the date range.
archive_override = os.environ.get('VISITOR_ARCHIVE')
if archive_override:
    database_path = Path(archive_override).expanduser().resolve()
else:
    candidates = [path for parent in [Path.cwd(), *Path.cwd().parents]
                  for path in [parent / 'visits.duckdb', parent / 'artifacts/visitor-access/visits.duckdb']]
    database_path = next((path for path in candidates if path.is_file()), None)
    if database_path is None:
        raise FileNotFoundError('Run from the Genealogy repository or notebook directory, or set VISITOR_ARCHIVE.')
output_directory = database_path.parent
figure_directory = output_directory / 'visit_analysis_figures'
figure_directory.mkdir(exist_ok=True)
connection = duckdb.connect(str(database_path), read_only=True)
columns = ['id', 'email', 'occurred_at', 'kind', 'outcome', 'path', 'timezone',
           'utc_offset_minutes', 'language', 'accept_language', 'ip_address', 'source_url']
rows = [dict(zip(columns, values)) for values in connection.execute(
    'SELECT ' + ','.join(columns) + ' FROM visitor_events ORDER BY occurred_at,id').fetchall()]
archive_info = connection.execute(
    'SELECT min(archived_at)::VARCHAR, max(archived_at)::VARCHAR FROM visitor_events').fetchone()
tables = {item[0] for item in connection.execute('SHOW TABLES').fetchall()}
legacy_import = connection.execute(
    'SELECT count(*),count(DISTINCT source_sha256) FROM legacy_csv_events').fetchone() if 'legacy_csv_events' in tables else (0, 0)
legacy_added = sum(row['source_url'].startswith('legacy-csv:') for row in rows)
connection.close()
if not rows:
    raise ValueError('The archive has no events to summarize.')
for row in rows:
    row['local_time'] = datetime.fromtimestamp(row['occurred_at'], timezone.utc).astimezone(TIMEZONE)
    row['email_key'] = (row['email'] or '').strip().lower()
first_time = min(row['local_time'] for row in rows)
last_time = max(row['local_time'] for row in rows)
date_label = f'{first_time:%b %d, %Y}–{last_time:%b %d, %Y} · America/Chicago'
dates = [first_time.date() + timedelta(days=i)
         for i in range((last_time.date() - first_time.date()).days + 1)]

COLORS = {'content': '#32687f', 'visits': '#426c3b', 'noise': '#a48a4b',
          'invalid': '#a0524f', 'muted': '#87949a', 'dark': '#273b43'}
plt.rcParams.update({'figure.figsize': (11, 4.5), 'figure.dpi': 130, 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.edgecolor': '#b8c1c5', 'axes.labelcolor': COLORS['dark'],
    'text.color': COLORS['dark'], 'xtick.color': COLORS['dark'], 'ytick.color': COLORS['dark'],
    'grid.color': '#e5e9eb', 'grid.linewidth': 0.7, 'savefig.facecolor': 'white'})
display(HTML('<style>.visit-table{border-collapse:collapse;width:100%;font-size:13px;margin:12px 0 24px}'
    '.visit-table th,.visit-table td{padding:8px 10px;border-bottom:1px solid #dce2e5;text-align:left;'
    'vertical-align:top;overflow-wrap:anywhere;max-width:360px}.visit-table th{background:#edf2f4;'
    'font-weight:600}.visit-table caption{text-align:left;font-weight:600;padding:8px 0;caption-side:top}</style>'))
def table(headers, records, caption=''):
    records = list(records)
    def value(item):
        if item is None: return '—'
        if isinstance(item, int): return f'{item:,}'
        return str(item)
    markup = '<table class="visit-table"><caption>' + html.escape(caption) + '</caption><thead><tr>'
    markup += ''.join('<th>' + html.escape(str(header)) + '</th>' for header in headers) + '</tr></thead><tbody>'
    if not records:
        markup += f'<tr><td colspan="{len(headers)}">No matching events in the archive.</td></tr>'
    for record in records:
        markup += '<tr>' + ''.join('<td>' + html.escape(value(item)) + '</td>' for item in record) + '</tr>'
    display(HTML(markup + '</tbody></table>'))
def chart(fig, name):
    fig.suptitle(date_label, fontsize=10, color='#64747b', y=1.02)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight')
    payload = buffer.getvalue()
    (figure_directory / (name + '.png')).write_bytes(payload)
    display(Image(data=payload, alt=name.replace('_', ' ')))
    plt.close(fig)
def date_axis(axis):
    interval = max(1, len(dates) // 10)
    axis.xaxis.set_major_locator(mdates.DayLocator(interval=interval))
    axis.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    axis.set_xlim(datetime.combine(dates[0], datetime.min.time()) - timedelta(hours=12),
                  datetime.combine(dates[-1], datetime.min.time()) + timedelta(hours=12))
    axis.grid(axis='y', zorder=0)
    axis.set_axisbelow(True)
    axis.yaxis.set_major_locator(MaxNLocator(integer=True))

In [ ]:
# Definitions are mutually exclusive for the event taxonomy; security flags may overlap it.
def content_class(row):
    if row['outcome'] == 'allowed' and row['kind'] == 'page':
        return 'Embedded app request' if '/app/' in row['path'] else 'Top-level page request'
    if row['outcome'] == 'allowed' and row['kind'] == 'download':
        return 'Data/API request' if row['path'].startswith('/api/') else 'Content download'
    if row['kind'] == 'challenge': return 'Access-screen redirect'
    if row['kind'] == 'entry': return 'Anonymous entry screen'
    if row['kind'] == 'access_attempt': return 'Login attempt'
    return 'Other logged event'

def probe_category(path):
    path = unquote(path).lower()
    if re.search(r'(^|/)\.env(?:[./]|$)', path): return 'Environment-file probe'
    if re.search(r'(^|/)\.git(?:/|$)', path): return 'Git repository probe'
    if re.search(r'(^|/)(?:wp-login\.php|wp-json|wp-includes|wp-admin|xmlrpc\.php)(?:/|$)', path):
        return 'WordPress/XML-RPC probe'
    return None

INVALID_OUTCOMES = {'rejected', 'forbidden', 'invalid_input', 'rate_limited'}
ERROR_OUTCOMES = {'not_found', 'error', 'server_error', 'internal_error', 'unavailable'}
for row in rows:
    row['activity'] = content_class(row)
    row['probe_category'] = probe_category(row['path'])
    row['invalid_attempt'] = row['kind'] == 'access_attempt' and row['outcome'] in INVALID_OUTCOMES
    row['logged_error'] = row['outcome'] in ERROR_OUTCOMES
    row['content_activity'] = row['activity'] in {'Top-level page request', 'Content download'}
    row['qa_email'] = row['email_key'].endswith('@example.invalid')

pages = [row for row in rows if row['activity'] == 'Top-level page request']
content_rows = [row for row in rows if row['content_activity']]
embedded_rows = [row for row in rows if row['activity'] == 'Embedded app request']
invalid_rows = [row for row in rows if row['invalid_attempt']]
probe_rows = [row for row in rows if row['probe_category']]
error_rows = [row for row in rows if row['logged_error']]
security_rows = [row for row in rows if row['invalid_attempt'] or row['probe_category']]

# Approximate visits from content activity only; no redirect or login noise enters this metric.
visits = []
latest_visit = {}
for row in content_rows:
    identity = (row['email_key'], row['ip_address'] or '[missing IP]')
    visit = latest_visit.get(identity)
    if visit is None or row['occurred_at'] - visit['last_epoch'] > VISIT_GAP_MINUTES * 60:
        visit = {'email': row['email_key'], 'ip': row['ip_address'], 'start': row['local_time'],
                 'last_epoch': row['occurred_at'], 'pages': 0, 'downloads': 0}
        visits.append(visit)
        latest_visit[identity] = visit
    visit['last_epoch'] = row['occurred_at']
    visit['pages'] += row['activity'] == 'Top-level page request'
    visit['downloads'] += row['activity'] == 'Content download'
identities = sorted({row['email_key'] for row in content_rows if row['email_key']})
activity_counts = Counter(row['activity'] for row in rows)
kind_outcomes = Counter((row['kind'], row['outcome']) for row in rows)
owner_pages = sum(row['email_key'] == OWNER_EMAIL for row in pages)
first_date, last_date = first_time.date(), last_time.date()
assert len({row['id'] for row in rows}) == len(rows), 'Duplicate event IDs need investigation.'
assert sum(activity_counts.values()) == len(rows)
assert len(pages) + len(embedded_rows) == sum(row['kind'] == 'page' and row['outcome'] == 'allowed' for row in rows)
assert len(content_rows) == len(pages) + activity_counts['Content download']
assert sum(visit['pages'] for visit in visits) == len(pages)
assert sum(visit['downloads'] for visit in visits) == activity_counts['Content download']

In [ ]:
display(Markdown(f"**{first_time:%b %d}–{last_time:%b %d, %Y} · Chicago time** · "
    f"{len(rows):,} archived events · **{len(visits):,} estimated visits** · "
    f"**{len(pages):,} top-level page requests** · **{len(identities)} content email identities**. "
    f"The owner accounts for {owner_pages / max(1,len(pages)):.1%} of page requests. First and last days are partial."))

## Visits and visitors

In [ ]:
daily_pages=Counter(row['local_time'].date() for row in pages)
daily_visits=Counter(visit['start'].date() for visit in visits)
fig,axes=plt.subplots(2,1,figsize=(10,5.3),sharex=True)
axes[0].bar(dates,[daily_pages[d] for d in dates],color=COLORS['content'])
axes[0].set(title='Top-level content requests',ylabel='Page requests')
axes[1].plot(dates,[daily_visits[d] for d in dates],'-o',color=COLORS['visits'])
axes[1].set(title='Estimated visits · 30-minute inactivity gap',ylabel='Visits')
for ax in axes:
    date_axis(ax);ax.set_ylim(bottom=0)
chart(fig,'01_daily_content')
visitor_table=[]
for email in identities:
    selected=[row for row in content_rows if row['email_key']==email]
    visitor_table.append((email,'Owner' if email==OWNER_EMAIL else 'Other',
        sum(v['email']==email for v in visits),
        sum(r['activity']=='Top-level page request' for r in selected),
        sum(r['activity']=='Content download' for r in selected),
        max(r['local_time'] for r in selected).strftime('%Y-%m-%d %H:%M %Z')))
visitor_table.sort(key=lambda item:-item[3])
table(['Recorded email','Role','Est. visits','Pages','Downloads','Most recent access (Chicago)'],visitor_table)
page_counts=Counter(row['path'] for row in pages)
table(['Most requested pages','Requests'],page_counts.most_common(5))

## Network sources

In [ ]:
network_cache_path = output_directory / 'network_sources.json'
network_cache = json.loads(network_cache_path.read_text(encoding='utf-8')) if network_cache_path.is_file() else {'records': []}
registration_records = [record for record in network_cache['records'] if record.get('start_address')]
def registered_provider(record):
    text = (record.get('registered_holder', '') + ' ' + record.get('network_name', '')).lower()
    for needle, label in [('cloudflare', 'Cloudflare'), ('comcast', 'Comcast'), ('google', 'Google'),
                          ('palo alto', 'Palo Alto Networks'), ('censys', 'Censys'), ('amazon', 'Amazon'),
                          ('m247', 'M247'), ('techties', 'TechTies'), ('contabo', 'Contabo'),
                          ('virtualine', 'Virtualine Technologies'), ('asianet', 'Asianet'),
                          ('modat', 'Modat'), ('unus', 'Unus')]:
        if needle in text: return label
    return record.get('registered_holder') or record.get('network_name') or 'Unclassified'
def registration_for(address):
    if not address: return None
    ip = ipaddress.ip_address(address)
    matches = [record for record in registration_records
               if ip.version == ipaddress.ip_address(record['start_address']).version
               and ipaddress.ip_address(record['start_address']) <= ip <= ipaddress.ip_address(record['end_address'])]
    return min(matches, key=lambda record: int(ipaddress.ip_address(record['end_address'])) - int(ipaddress.ip_address(record['start_address']))) if matches else None
ip_registration = {address: registration_for(address) for address in {row['ip_address'] for row in rows}}
for row in rows:
    registration = ip_registration.get(row['ip_address'])
    row['provider'] = registered_provider(registration) if registration else 'Unclassified / lookup unavailable'
providers = sorted({row['provider'] for row in rows}, key=lambda provider: -sum(row['provider'] == provider for row in rows))
network_summary = []
for provider in providers:
    selected = [row for row in rows if row['provider'] == provider]
    network_summary.append((provider, len({row['ip_address'] for row in selected}), len(selected),
        sum(row['activity'] == 'Top-level page request' for row in selected),
        sum(row['kind'] == 'challenge' for row in selected), sum(row['invalid_attempt'] for row in selected),
        sum(bool(row['probe_category']) for row in selected)))

In [ ]:
leading=network_summary[:5]
if len(network_summary)>5:
    remainder=network_summary[5:]
    leading.append(('Other registered holders',sum(x[1] for x in remainder),
        sum(x[2] for x in remainder),sum(x[3] for x in remainder),
        sum(x[4] for x in remainder),sum(x[5] for x in remainder),sum(x[6] for x in remainder)))
fig,axes=plt.subplots(1,2,figsize=(10,3.3),sharey=True)
labels=[r[0] for r in leading][::-1]
axes[0].barh(labels,[r[2] for r in leading][::-1],color=COLORS['muted'])
axes[0].set(title='All events',xlabel='Events')
axes[1].barh(labels,[r[3] for r in leading][::-1],color=COLORS['content'])
axes[1].set(title='Content pages',xlabel='Page requests')
for ax in axes:
    ax.grid(axis='x');ax.set_axisbelow(True);ax.xaxis.set_major_locator(MaxNLocator(integer=True))
chart(fig,'02_network_sources')
table(['Registered holder','Source IPs','All events','Pages','Probe requests'],
      [(r[0],r[1],r[2],r[3],r[6]) for r in leading])
ip_totals=Counter(row['ip_address'] for row in rows)
leading_ip,leading_count=ip_totals.most_common(1)[0]
leading_ip_rows=[r for r in rows if r['ip_address']==leading_ip]
display(Markdown(f"**{leading_ip}** generated {leading_count:,} events and "
    f"{sum(r['content_activity'] for r in leading_ip_rows):,} content events. "
    "Network holders can be proxies or relays; they do not identify the visitor or physical location."))

## Invalid access and errors

In [ ]:
probe_counts=Counter(r['probe_category'] for r in probe_rows)
invalid_counts=Counter(r['outcome'] for r in invalid_rows)
qa_failures=sum(r['qa_email'] for r in invalid_rows)
probe_outcomes=Counter(r['outcome'] for r in probe_rows)
blocked=sum(r['outcome'] in {'denied','rejected','forbidden'} for r in probe_rows)
display(Markdown(f"**{len(probe_rows):,} probe-pattern requests** ({blocked:,} denied/rejected/forbidden) · "
    f"**{len(invalid_rows):,} unsuccessful login attempts**, including {qa_failures:,} explicit QA attempts · "
    f"**{len(error_rows):,} explicitly logged errors**. "
    f"The {activity_counts['Access-screen redirect']:,} access redirects are excluded from usage; a redirect alone does not imply an attack."))
fig,ax=plt.subplots(figsize=(10,2.8))
items=probe_counts.most_common()[::-1]
if items:
    bars=ax.barh([r[0] for r in items],[r[1] for r in items],color=COLORS['invalid'])
    ax.bar_label(bars,padding=4);ax.margins(x=.15)
else:
    ax.text(.5,.5,'No explicit probe patterns found',ha='center',transform=ax.transAxes)
ax.set(title='Requests for configuration, repository, and WordPress paths',xlabel='Requests')
ax.xaxis.set_major_locator(MaxNLocator(integer=True));ax.grid(axis='x');ax.set_axisbelow(True)
chart(fig,'03_invalid_access')
error_paths=Counter(r['path'] for r in error_rows)
error_example=', '.join(f'{path} ({count})' for path,count in error_paths.most_common(3)) or 'None logged'
table(['Signal','Events','Detail'],[
    ('Rejected email submissions',invalid_counts['rejected'],'May include mistakes or testing'),
    ('Forbidden login submissions',invalid_counts['forbidden'],'Failed origin checks; intent unknown'),
    ('Invalid input / rate limited',invalid_counts['invalid_input']+invalid_counts['rate_limited'],'Explicit gateway outcomes'),
    ('Logged errors',len(error_rows),error_example),
])
display(Markdown('The event log does not record every HTTP failure, so it cannot establish a complete error rate or prove successful delivery.'))

In [ ]:
# Show rejected submissions explicitly; origin failures have no submitted email.
rejected_by_email = defaultdict(list)
for row in rows:
    if row['kind'] == 'access_attempt' and row['outcome'] == 'rejected':
        rejected_by_email[row['email_key'] or '[not supplied]'].append(row)
rejected_email_table = [
    (address, len(attempts), max(row['local_time'] for row in attempts).strftime('%Y-%m-%d %H:%M %Z'),
     'Yes' if address.endswith('@example.invalid') else 'No')
    for address, attempts in rejected_by_email.items()
]
rejected_email_table.sort(key=lambda record: (-record[1], record[0]))
table(['Rejected email address', 'Attempts', 'Most recent rejection (Chicago)', 'Explicit QA address'],
      rejected_email_table, 'Rejected email submissions · full archive')
assert sum(record[1] for record in rejected_email_table) == invalid_counts['rejected']

In [ ]:
from IPython.utils.capture import capture_output
# Keep supporting evidence available without expanding the main report.
with capture_output() as captured:
    table(['Kind','Outcome','Events'],[(k,o,n) for (k,o),n in kind_outcomes.most_common()],'Full event inventory')
    table(['Chicago time','Email','Source IP','Outcome'],
        [(r['local_time'].isoformat(timespec='seconds'),r['email_key'] or '[not supplied]',r['ip_address'],r['outcome']) for r in invalid_rows],
        'Unsuccessful login details')
    table(['Probe pattern','Path','Outcome','Requests'],
        [(category,path,outcome,count) for (category,path,outcome),count in Counter(
            (r['probe_category'],r['path'],r['outcome']) for r in probe_rows).most_common()],
        'Matched probe paths')
    table(['Registered name','Allocation range','Retrieved UTC','Registry source'],
        [(r.get('network_name','')+' / '+r.get('registered_holder',''),
          r['start_address']+' – '+r['end_address'],r['retrieved_at_utc'],r['source_url']) for r in registration_records],
        'RDAP evidence')
    unmatched=[address for address,record in ip_registration.items() if record is None]
    if unmatched:
        table(['Unclassified source IP','Events'],[(ip,ip_totals[ip]) for ip in unmatched],'Unresolved network sources')
methodology=(
    '<p><strong>Scope:</strong> Entire visitor_events table, with no date restriction. Rerun after export; later online events are not yet included. Dates use America/Chicago.</p>'
    '<p><strong>Estimated visit:</strong> Content events sharing email and exact IP, separated by no more than 30 minutes. IP changes can split visits; shared devices can merge them. Emails are claimed identities, not verified people.</p>'
    f'<p><strong>Usage exclusions:</strong> {len(embedded_rows):,} embedded /app/ requests and {activity_counts["Data/API request"]:,} API data requests are reported separately from top-level content interest. Reloads still count. Directly opened app paths are also excluded because frame/referrer data is unavailable.</p>'
    '<p><strong>Security:</strong> Probe rules match .env, .git, WordPress and XML-RPC paths. They overlap redirect events, so do not add them to event totals. Failed logins may be mistakes or QA. HTTP status, method, user agent, host and session IDs are absent; some HTTP errors are unlogged.</p>'
    f'<p><strong>Preserved history:</strong> {legacy_added:,} older events were recovered from the removed CSV, with {legacy_import[0]:,} original rows retained in legacy_csv_events as provenance. They are not counted twice. The original CSV did not identify its request host.</p>'
    '<p><strong>Networks:</strong> Cached network_sources.json stores regional registry evidence. Registered holders are not verified visitors, geographic locations, or bot identities. New unmatched addresses remain unclassified. Rerunning does not issue network lookups or modify the database.</p>'
    f'<p><strong>Archive:</strong> {html.escape(str(database_path))}; latest archive timestamp {html.escape(str(archive_info[1]))}.</p>'
)
content=''.join(output.data.get('text/html','') for output in captured.outputs)
display(HTML('<details><summary><strong>Definitions, provenance, and supporting tables</strong></summary>'+methodology+content+'</details>'))
assert sum(r[2] for r in network_summary)==len(rows)
assert sum(r[3] for r in network_summary)==len(pages)
assert sum(v['pages'] for v in visits)==len(pages)
assert sum(v['downloads'] for v in visits)==activity_counts['Content download']
assert sum(daily_visits.values())==len(visits)
assert sum(r[3] for r in visitor_table)==len(pages)